## Imports

In [ ]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler, FunctionTransformer
from sklearn.impute import SimpleImputer

## Load Data

In [52]:
df = pd.read_csv("../raw_data/hotel_bookings.csv")

## Engineer Numerical Features

In [ ]:
def engineer_numerical_features(X: pd.DataFrame) -> pd.DataFrame:
    """
    Create engineered features from the raw hotel booking dataset.

    This function:
    - removes the target if it is present
    - removes known leakage columns
    - converts ID-like columns into binary indicators
    - creates additional behavioral features

    Parameters
    ----------
    X : pd.DataFrame
        Raw input dataframe.

    Returns
    -------
    pd.DataFrame
        Dataframe with engineered features added.
    """
    X = X.copy()

    # Drop target if it is accidentally included
    if "is_canceled" in X.columns:
        X = X.drop(columns="is_canceled")

    # Drop leakage columns: these contain future information
    leakage_cols = ["reservation_status", "reservation_status_date"]
    cols_to_drop = [col for col in leakage_cols if col in X.columns]
    if cols_to_drop:
        X = X.drop(columns=cols_to_drop)

    # company ID -> binary flag
    if "company" in X.columns:
        X["company_booking"] = X["company"].notna().astype(int)
        X = X.drop(columns="company")

    # agent ID -> binary flag
    if "agent" in X.columns:
        X["has_agent"] = X["agent"].notna().astype(int)
        X = X.drop(columns="agent")

    # parking spaces -> simpler yes/no feature
    if "required_car_parking_spaces" in X.columns:
        X["has_parking"] = (X["required_car_parking_spaces"] > 0).astype(int)
        X = X.drop(columns="required_car_parking_spaces")

    # children / babies flags
    if "children" in X.columns:
        X["has_children"] = (X["children"].fillna(0) > 0).astype(int)

    if "babies" in X.columns:
        X["has_babies"] = (X["babies"].fillna(0) > 0).astype(int)

    # family booking flag
    if {"children", "babies"}.issubset(X.columns):
        X["is_family"] = (X[["children", "babies"]].fillna(0).sum(axis=1) > 0).astype(int)

    # total stay length
    if {"stays_in_weekend_nights", "stays_in_week_nights"}.issubset(X.columns):
        X["total_stay"] = (
            X["stays_in_weekend_nights"] + X["stays_in_week_nights"]
        )

    # total guests
    if {"adults", "children", "babies"}.issubset(X.columns):
        X["total_guests"] = (X["adults"] + X["children"].fillna(0) + X["babies"].fillna(0))

    return X

## Define Columns

In [59]:
numeric_features = [
    "lead_time",
    "arrival_date_year",
    "arrival_date_week_number",
    "arrival_date_day_of_month",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adults",
    "children",
    "babies",
    "previous_cancellations",
    "previous_bookings_not_canceled",
    "booking_changes",
    "days_in_waiting_list",
    "adr",
    "total_of_special_requests",
    "total_stay",
    "total_guests"
]

binary_features = [
    "is_repeated_guest",
    "company_booking",
    "has_agent",
    "has_parking",
    "has_children",
    "has_babies",
    "is_family"
]

categorical_features = [
    "hotel",
    "arrival_date_month",
    "meal",
    "country",
    "market_segment",
    "distribution_channel",
    "reserved_room_type",
    "assigned_room_type",
    "deposit_type",
    "customer_type"
]

## Preprocess Pipeline Numerical and Binary Features 

In [62]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])

binary_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent"))
])

column_preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("bin", binary_pipeline, binary_features),
])

preproc_pipeline = Pipeline([
    ("feature_engineering", FunctionTransformer(engineer_hotel_features, validate=False)),
    ("preprocessing", column_preprocessor)
])